# Deteksi Penyakit Paru-Paru — Citra USG
**Model:** RT-DETR (ResNet-50) · **Framework:** Ultralytics

| Kelas | ID |
|---|---|
| Kavitas | 0 |
| Pola B | 1 |
| Konsolidasi | 2 |
| Penebalan Pleura | 3 |
| Bullae | 4 |
| Infiltrat Terkonsolidasi | 5 |

> **Split:** Stratified per kelas (70/20/10). Dipilih karena beberapa kelas hanya ada pada 1 pasien — wajib dicatat sebagai limitasi dalam laporan.


## 1. Install & Import

In [1]:
import os, shutil, random, yaml, cv2, re
from pathlib import Path
from collections import defaultdict
from multiprocessing.pool import ThreadPool
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from ultralytics import RTDETR

random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

GPU: NVIDIA GeForce RTX 3060


## 2. Konfigurasi

In [3]:
from pathlib import Path

BASE          = Path('G:/My Drive/Dataset Riset Makcik')

JPG_ROOT      = BASE / 'JPG'
ANOTASI_ROOT  = BASE / 'Anotasi'
LUS_BALD_ROOT = BASE / 'LUS-BALD'
NORMAL_ROOT   = JPG_ROOT / 'Normal'

# OUTPUT KE SSD LOKAL
OUTPUT_ROOT = Path(r'D:\RIZ\Training')

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CLASS_NAMES = [
    'kavitas',
    'pola_b',
    'konsolidasi',
    'penebalan_pleura',
    'bullae',
    'infiltrat_terkonsolidasi',
    'pleura_ireguler'
]

NC = len(CLASS_NAMES)

SESSION_TO_PATIENT = {
    '0404':'04',

    '0601':'06',
    '0604':'06',
    '0605':'06',

    '0701':'07',
    '0702':'07',
    '0704':'07',

    '0801':'08',
    '0802':'08',

    '0901':'09',
    '0902':'09',
    '0903':'09',

    '1103':'11',
    '1105':'11',
    '1106':'11',
    '1107':'11',
    '1108':'11',
    '1112':'11',

    '1203':'12',
    '1204':'12',
    '1219':'12',
    '1220':'12',

    '1307':'13',
    '1315':'13',

    '1525':'15',
    '1526':'15',

    '1611':'16',
    '1613':'16',

    '1908':'19',
    '1910':'19',
    '1914':'19',
    '1915':'19',
    '1916':'19',

    '2002':'20',
    '2003':'20',
    '2005':'20',

    '2105':'21',
    '2107':'21',
    '2109':'21',
    '2115':'21',

    '2211':'22',
    '2212':'22',
    '2215':'22',
    '2218':'22',
    '2228':'22',
    '2230':'22',
}

PREFIXED_PATIENTS = {'11'}

SPLIT_RATIO  = (0.70, 0.20, 0.10)

BG_RATIO     = 0.5

NORMAL_RATIO = 1.0

for label, p in [
    ('JPG_ROOT', JPG_ROOT),
    ('ANOTASI_ROOT', ANOTASI_ROOT),
    ('LUS_BALD_ROOT', LUS_BALD_ROOT),
    ('NORMAL_ROOT', NORMAL_ROOT)
]:

    status = '✓' if p.exists() else '✗ TIDAK ADA'

    n = len(list(p.iterdir())) if p.exists() else 0

    print(f'  {status} {label}: {p}  ({n} item)')

print(f'\nOUTPUT_ROOT: {OUTPUT_ROOT}')

  ✓ JPG_ROOT: G:\My Drive\Dataset Riset Makcik\JPG  (47 item)
  ✓ ANOTASI_ROOT: G:\My Drive\Dataset Riset Makcik\Anotasi  (46 item)
  ✓ LUS_BALD_ROOT: G:\My Drive\Dataset Riset Makcik\LUS-BALD  (3 item)
  ✓ NORMAL_ROOT: G:\My Drive\Dataset Riset Makcik\JPG\Normal  (21 item)

OUTPUT_ROOT: D:\RIZ\Training


## 3. Scan & Validasi Dataset

In [4]:
def build_txt_index(anotasi_dir, sid, pid):

    prefix = f'{pid}_JPG_{sid}_'

    idx = {}

    for txt in anotasi_dir.glob('*.txt'):

        stem = txt.stem

        # khusus pasien tertentu
        if (
            pid in PREFIXED_PATIENTS and
            stem.startswith(prefix)
        ):
            stem = stem[len(prefix):]

        m = re.search(r'(\d+)$', stem)

        key = str(int(m.group(1))) if m else stem

        idx[key] = txt

    return idx

# ── Ringkasan Dataset ─────────────────────────────────────────
print(
    f'{"Sesi":<8} '
    f'{"Pasien":<8} '
    f'{"JPG":<7} '
    f'{"Anotasi":<9} '
    f'{"BG":<6} '
    f'Kelas'
)

print('-' * 90)

global_bbox = defaultdict(int)

total_jpg = 0
total_annot = 0
total_bg = 0

for sid, pid in sorted(SESSION_TO_PATIENT.items()):

    jpg_dir = JPG_ROOT / sid
    ann_dir = ANOTASI_ROOT / sid

    if not jpg_dir.exists():

        print(
            f'{sid:<8} '
            f'{pid:<8} '
            f'✗ folder JPG tidak ada'
        )

        continue

    txt_idx = (
        build_txt_index(ann_dir, sid, pid)
        if ann_dir.exists()
        else {}
    )

    jpgs = sorted(jpg_dir.glob('*.jpg'))

    annot = 0
    bg = 0

    class_set = set()

    for jpg in jpgs:

        m = re.search(r'(\d+)$', jpg.stem)

        key = str(int(m.group(1))) if m else jpg.stem

        txt = txt_idx.get(key)

        # annotated
        if (
            txt and
            txt.exists() and
            txt.stat().st_size > 0
        ):

            try:

                lines = (
                    txt.read_text(
                        encoding='utf-8',
                        errors='ignore'
                    )
                    .strip()
                    .splitlines()
                )

            except:
                lines = []

            for line in lines:

                p = line.strip().split()

                if len(p) >= 5:

                    try:

                        cid = int(p[0])

                        if 0 <= cid < NC:

                            global_bbox[cid] += 1

                            class_set.add(cid)

                    except:
                        pass

            annot += 1

        # background
        else:
            bg += 1

    names = (
        ', '.join(
            CLASS_NAMES[c]
            for c in sorted(class_set)
        )
        if class_set
        else '— tidak ada'
    )

    print(
        f'{sid:<8} '
        f'{pid:<8} '
        f'{len(jpgs):<7} '
        f'{annot:<9} '
        f'{bg:<6} '
        f'{names}'
    )

    total_jpg += len(jpgs)
    total_annot += annot
    total_bg += bg

print('-' * 90)

print(
    f'{"Total":<8} '
    f'{"":8} '
    f'{total_jpg:<7} '
    f'{total_annot:<9} '
    f'{total_bg}'
)

# ── Distribusi Class ──────────────────────────────────────────
print('\nDistribusi bbox per kelas:')

tb = sum(global_bbox.values())

for cid, name in enumerate(CLASS_NAMES):

    n = global_bbox.get(cid, 0)

    pct = (n / tb * 100) if tb else 0

    bar = '█' * int(pct / 2)

    flag = (
        ' ← TIDAK ADA'
        if n == 0
        else ' ← sedikit'
        if n < 20
        else ''
    )

    print(
        f'  [{cid}] '
        f'{name:<30} '
        f'{n:>5} '
        f'({pct:4.1f}%) '
        f'{bar}{flag}'
    )

print(f'\n  Total bbox: {tb}')

# ── Output info ───────────────────────────────────────────────
print(f'\nOUTPUT_ROOT: {OUTPUT_ROOT}')

Sesi     Pasien   JPG     Anotasi   BG     Kelas
------------------------------------------------------------------------------------------
0404     04       123     19        104    konsolidasi
0601     06       121     5         116    kavitas
0604     06       125     13        112    kavitas
0605     06       166     1         165    kavitas
0701     07       101     10        91     pola_b
0702     07       160     9         151    konsolidasi
0704     07       101     10        91     konsolidasi
0801     08       95      8         87     konsolidasi
0802     08       93      29        64     pola_b, konsolidasi
0901     09       85      13        72     penebalan_pleura
0902     09       112     17        95     konsolidasi
0903     09       81      9         72     pola_b
1103     11       136     65        71     konsolidasi
1105     11       129     8         121    konsolidasi
1106     11       107     30        77     infiltrat_terkonsolidasi
1107     11       102     40   

## 4. Pipeline — Preprocessing + Stratified Split + YAML
Membaca **langsung dari Drive** (JPG & Anotasi), output ke `Drive/output/`.

In [5]:
# ── Bersihkan output lama ─────────────────────────────────────
for folder in ['preprocessed', 'split', 'runs']:

    p = OUTPUT_ROOT / folder

    if p.exists():
        shutil.rmtree(p)
        print(f'Dihapus: {p}')

# hapus cache ultralytics
for cache_file in BASE.rglob("*.cache"):

    try:
        cache_file.unlink()
        print(f'Hapus cache: {cache_file}')

    except:
        pass

# ── A. Preprocessing ──────────────────────────────────────────
print('\nA. Preprocessing (CLAHE + bilateral filter + resize 640)...')

TARGET_SIZE = 640

def preprocess_one(args):

    src_jpg, dst_img, src_txt, dst_lbl = args

    img = cv2.imread(str(src_jpg))

    if img is None:
        return 0

    # grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE
    enh = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    ).apply(gray)

    # bilateral filter
    filt = cv2.bilateralFilter(
        enh,
        9,
        75,
        75
    )

    # FIX IMAGE SIZE
    filt = cv2.resize(
        filt,
        (TARGET_SIZE, TARGET_SIZE),
        interpolation=cv2.INTER_AREA
    )

    # convert back to 3 channel
    out = cv2.cvtColor(
        filt,
        cv2.COLOR_GRAY2BGR
    )

    cv2.imwrite(str(dst_img), out)

    # labels
    if src_txt and src_txt.exists() and src_txt.stat().st_size > 0:
        shutil.copy2(src_txt, dst_lbl)
    else:
        dst_lbl.touch()

    return 1

tasks = []
frame_meta = []

for sid, pid in sorted(SESSION_TO_PATIENT.items()):

    jpg_dir = JPG_ROOT / sid
    ann_dir = ANOTASI_ROOT / sid

    if not jpg_dir.exists():
        continue

    txt_idx = build_txt_index(
        ann_dir,
        sid,
        pid
    ) if ann_dir.exists() else {}

    jpgs = sorted(jpg_dir.glob('*.jpg'))

    annotated = []
    background = []

    for jpg in jpgs:

        m = re.search(r'(\d+)$', jpg.stem)

        key = str(int(m.group(1))) if m else jpg.stem

        txt = txt_idx.get(key)

        if txt and txt.exists() and txt.stat().st_size > 0:
            annotated.append((jpg, txt, key))
        else:
            background.append((jpg, None, key))

    max_bg = max(
        int(len(annotated) * BG_RATIO),
        5
    )

    bg_sel = random.sample(
        background,
        min(max_bg, len(background))
    )

    selected = annotated + bg_sel

    print(
        f'  [{sid}] '
        f'annot={len(annotated)} '
        f'bg={len(bg_sel)}/{len(background)}'
    )

    dst_img_dir = OUTPUT_ROOT / 'preprocessed' / sid / 'images'
    dst_lbl_dir = OUTPUT_ROOT / 'preprocessed' / sid / 'labels'

    dst_img_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    dst_lbl_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for jpg, txt, key in selected:

        dst_img = dst_img_dir / jpg.name
        dst_lbl = dst_lbl_dir / (jpg.stem + '.txt')

        tasks.append(
            (
                jpg,
                dst_img,
                txt,
                dst_lbl
            )
        )

        class_ids = set()

        if txt and txt.exists() and txt.stat().st_size > 0:

            for line in txt.read_text().strip().splitlines():

                p = line.strip().split()

                if len(p) >= 5:

                    try:
                        cid = int(p[0])

                        if 0 <= cid < NC:
                            class_ids.add(cid)

                    except:
                        pass

        frame_meta.append(
            (
                sid,
                jpg.stem,
                bool(class_ids),
                class_ids
            )
        )

print(f'  Total frame: {len(tasks)}')

with ThreadPool(2) as pool:

    done = list(
        pool.imap_unordered(
            preprocess_one,
            tasks
        )
    )

print(f'  Selesai: {sum(done)}/{len(tasks)}')

# ── B. Stratified split ───────────────────────────────────────
print('\nB. Stratified split 70/20/10 per kelas...')

annotated_frames = [
    (s,st,cids)
    for s,st,has,cids in frame_meta
    if has
]

background_frames = [
    (s,st)
    for s,st,has,cids in frame_meta
    if not has
]

total_annotated = len(annotated_frames)

max_normal = max(
    int(total_annotated * NORMAL_RATIO),
    10
)

print(
    f'  Total annotated: {total_annotated} '
    f'→ max Normal bg: {max_normal}'
)

frames_by_class = defaultdict(list)

for s,st,cids in annotated_frames:

    for c in cids:
        frames_by_class[c].append((s,st))

frame_split = {}

for cid in range(NC):

    frames = list(
        dict.fromkeys(
            frames_by_class[cid]
        )
    )

    random.shuffle(frames)

    n = len(frames)

    if n == 0:
        continue

    n_test = max(
        1,
        round(n * SPLIT_RATIO[2])
    ) if n >= 3 else 0

    n_val = max(
        1,
        round(n * SPLIT_RATIO[1])
    ) if n >= 2 else 0

    n_train = n - n_val - n_test

    for j, key in enumerate(frames):

        if key not in frame_split:

            if j < n_train:
                frame_split[key] = 'train'

            elif j < n_train + n_val:
                frame_split[key] = 'val'

            else:
                frame_split[key] = 'test'

for s,st,cids in annotated_frames:

    if (s,st) not in frame_split:
        frame_split[(s,st)] = 'train'

random.shuffle(background_frames)

nb_ = len(background_frames)

nt_ = max(
    1,
    round(nb_ * SPLIT_RATIO[2])
) if nb_ >= 3 else 0

nv_ = max(
    1,
    round(nb_ * SPLIT_RATIO[1])
) if nb_ >= 2 else 0

for j,(s,st) in enumerate(background_frames):

    if j < nb_ - nv_ - nt_:
        frame_split[(s,st)] = 'train'

    elif j < nb_ - nt_:
        frame_split[(s,st)] = 'val'

    else:
        frame_split[(s,st)] = 'test'

# ── C. Salin ke struktur YOLO ─────────────────────────────────
print('\nC. Salin ke split/...')

split_root = OUTPUT_ROOT / 'split'

for s in ['train','val','test']:

    (split_root/'images'/s).mkdir(
        parents=True,
        exist_ok=True
    )

    (split_root/'labels'/s).mkdir(
        parents=True,
        exist_ok=True
    )

for (sid,stem), sp in frame_split.items():

    src_img = OUTPUT_ROOT/'preprocessed'/sid/'images'/(stem+'.jpg')
    src_lbl = OUTPUT_ROOT/'preprocessed'/sid/'labels'/(stem+'.txt')

    if not src_img.exists():
        continue

    shutil.copy2(
        src_img,
        split_root/'images'/sp/f'{sid}_{stem}.jpg'
    )

    dst = split_root/'labels'/sp/f'{sid}_{stem}.txt'

    if src_lbl.exists():
        shutil.copy2(src_lbl, dst)
    else:
        dst.touch()

# ── D. LUS-BALD ───────────────────────────────────────────────
lus_n = 0

if LUS_BALD_ROOT.exists():

    li = LUS_BALD_ROOT/'train'/'images'
    ll = LUS_BALD_ROOT/'train'/'labels'

    if not li.exists():
        li = ll = LUS_BALD_ROOT/'train'

    for img in list(li.glob('*.jpg')) + list(li.glob('*.png')):

        shutil.copy2(
            img,
            split_root/'images'/'train'/f'lus_{img.name}'
        )

        ls = ll/(img.stem+'.txt')

        dl = split_root/'labels'/'train'/f'lus_{img.stem}.txt'

        if ls.exists():
            shutil.copy2(ls, dl)
        else:
            dl.touch()

        lus_n += 1

    print(f'  LUS-BALD → train: {lus_n} gambar')

# ── E. Normal dataset ─────────────────────────────────────────
normal_n = 0

if NORMAL_ROOT.exists():

    normal_imgs = []

    for patient_dir in sorted(NORMAL_ROOT.iterdir()):

        if not patient_dir.is_dir():
            continue

        for point_dir in sorted(patient_dir.iterdir()):

            if not point_dir.is_dir():
                continue

            for img in sorted(point_dir.glob('*.jpg')):
                normal_imgs.append(img)

    random.shuffle(normal_imgs)

    normal_imgs = normal_imgs[:max_normal]

    nn = len(normal_imgs)

    nt = max(
        1,
        round(nn * SPLIT_RATIO[2])
    ) if nn >= 3 else 0

    nv = max(
        1,
        round(nn * SPLIT_RATIO[1])
    ) if nn >= 2 else 0

    for j, img in enumerate(normal_imgs):

        if j < nn - nv - nt:
            sp = 'train'

        elif j < nn - nt:
            sp = 'val'

        else:
            sp = 'test'

        dst_img = split_root/'images'/sp/f'normal_{img.parent.name}_{img.name}'

        dst_lbl = split_root/'labels'/sp/f'normal_{img.parent.name}_{img.stem}.txt'

        raw = cv2.imread(str(img))

        if raw is not None:

            gray = cv2.cvtColor(
                raw,
                cv2.COLOR_BGR2GRAY
            )

            enh = cv2.createCLAHE(
                clipLimit=2.0,
                tileGridSize=(8,8)
            ).apply(gray)

            filt = cv2.bilateralFilter(
                enh,
                9,
                75,
                75
            )

            filt = cv2.resize(
                filt,
                (TARGET_SIZE, TARGET_SIZE),
                interpolation=cv2.INTER_AREA
            )

            out = cv2.cvtColor(
                filt,
                cv2.COLOR_GRAY2BGR
            )

            cv2.imwrite(str(dst_img), out)

        else:
            shutil.copy2(img, dst_img)

        dst_lbl.touch()

        normal_n += 1

print(f'  Normal background: {normal_n}')

# ── F. YAML ───────────────────────────────────────────────────
print('\nF. Buat YAML...')

yaml_cfg = {
    'path': str(split_root.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': NC,
    'names': CLASS_NAMES,
}

yaml_path = OUTPUT_ROOT / 'lung_usg.yaml'

with open(yaml_path, 'w', encoding='utf-8') as f:

    yaml.dump(
        yaml_cfg,
        f,
        default_flow_style=False,
        allow_unicode=True
    )

YAML_PATH = str(yaml_path)

print(f'\n✓ Pipeline selesai')
print(f'YAML_PATH = "{YAML_PATH}"')

Dihapus: D:\RIZ\Training\preprocessed
Dihapus: D:\RIZ\Training\split
Dihapus: D:\RIZ\Training\runs

A. Preprocessing (CLAHE + bilateral filter + resize 640)...
  [0404] annot=19 bg=9/104
  [0601] annot=5 bg=5/116
  [0604] annot=13 bg=6/112
  [0605] annot=1 bg=5/165
  [0701] annot=10 bg=5/91
  [0702] annot=9 bg=5/151
  [0704] annot=10 bg=5/91
  [0801] annot=8 bg=5/87
  [0802] annot=29 bg=14/64
  [0901] annot=13 bg=6/72
  [0902] annot=17 bg=8/95
  [0903] annot=9 bg=5/72
  [1103] annot=65 bg=32/71
  [1105] annot=8 bg=5/121
  [1106] annot=30 bg=15/77
  [1107] annot=40 bg=20/62
  [1108] annot=27 bg=13/94
  [1112] annot=74 bg=37/61
  [1203] annot=12 bg=6/95
  [1204] annot=15 bg=7/62
  [1219] annot=5 bg=5/65
  [1220] annot=8 bg=5/115
  [1307] annot=13 bg=6/104
  [1315] annot=11 bg=5/62
  [1525] annot=43 bg=21/87
  [1526] annot=16 bg=8/93
  [1611] annot=15 bg=7/99
  [1613] annot=15 bg=7/101
  [1908] annot=10 bg=5/170
  [1910] annot=16 bg=8/152
  [1914] annot=11 bg=5/99
  [1915] annot=9 bg=5/16

## 5. Training RT-DETR

In [6]:
import psutil
from pathlib import Path
from ultralytics import RTDETR

# ── RAM INFO ──────────────────────────────────────────────────
ram = psutil.virtual_memory()

print(
    f'RAM: '
    f'{ram.used/1e9:.1f}GB / '
    f'{ram.total/1e9:.1f}GB '
    f'({ram.percent}%)'
)

# ── CHECK YAML ────────────────────────────────────────────────
assert (
    'YAML_PATH' in globals() and
    Path(YAML_PATH).exists()
), 'Jalankan Cell 4 dulu!'

RUNS_DIR = OUTPUT_ROOT / 'runs'

RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f'YAML  : {YAML_PATH}')
print(f'Output: {RUNS_DIR}')

# ── HAPUS CACHE ULTRALYTICS ───────────────────────────────────
for cache_file in OUTPUT_ROOT.rglob("*.cache"):

    try:
        cache_file.unlink()
        print(f'Hapus cache: {cache_file}')

    except:
        pass

# ── BASE CONFIG ───────────────────────────────────────────────
BASE_CFG = dict(

    data=YAML_PATH,

    imgsz=640,

    batch=4,

    workers=0,

    cache=False,

    rect=False,

    optimizer='AdamW',

    weight_decay=1e-4,

    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.4,

    degrees=10.0,

    translate=0.1,

    scale=0.3,

    fliplr=0.5,

    flipud=0.0,

    copy_paste=0.0,

    mixup=0.0,

    multi_scale=False,

    dropout=0.2,

    plots=True,

    save=True,

    save_period=20,

    project=str(RUNS_DIR),

    exist_ok=True,

    seed=42,

    deterministic=True,

    verbose=True,
)

# ── TRAIN TAHAP 1 ─────────────────────────────────────────────
print('\n[Tahap 1] Freeze backbone — 30 epoch...')

model = RTDETR('rtdetr-l.pt')

model.train(

    **BASE_CFG,

    epochs=30,

    freeze=10,

    lr0=5e-4,

    lrf=0.01,

    warmup_epochs=5,

    mosaic=0.0,

    close_mosaic=0,

    patience=30,

    amp=True,

    name='lung_s1',
)

print('\n✓ Tahap 1 selesai')

RAM: 13.4GB / 34.2GB (39.2%)
YAML  : D:\RIZ\Training\lung_usg.yaml
Output: D:\RIZ\Training\runs

[Tahap 1] Freeze backbone — 30 epoch...
New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.12.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\RIZ\Training\lung_usg.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.01, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=N

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      2.26G      1.451      6.393      3.322          3        640: 100% ━━━━━━━━━━━━ 391/391 3.2it/s 2:00<0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 6.5it/s 7.9s0.1s
                   all        404        166   2.24e-05     0.0823   0.000233   3.93e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30      2.53G      1.012     0.9097      2.154          4        640: 0% ──────────── 0/391  0.3s

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      2.53G      1.403     0.4874      2.662          3        640: 100% ━━━━━━━━━━━━ 391/391 3.8it/s 1:43<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.7it/s 6.6s0.1s
                   all        404        166          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30      2.53G     0.8738      2.447     0.8975          4        640: 0% ──────────── 0/391  0.3s

...

      28/30      2.54G      1.837   0.000216      2.119          3        640: 100% ━━━━━━━━━━━━ 391/391 3.8it/s 1:42<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.6it/s 6.7s0.1s
                   all        404        166          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/30      2.54G      2.016  0.0001327       1.44          4        640: 0% ──────────── 0/391  0.2s

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      2.54G      1.892  0.0002007       2.16          3        640: 100% ━━━━━━━━━━━━ 391/391 3.8it/s 1:43<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.6it/s 6.7s0.1s
                   all        404        166          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30      2.54G      2.045  0.0001364      2.088          4        640: 0% ──────────── 0/391  0.3s

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      2.54G      1.884  0.0002065      2.133          3        640: 100% ━━━━━━━━━━━━ 391/391 3.7it/s 1:44<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.7it/s 6.7s0.1s
                   all        404        166          0          0          0          0

30 epochs completed in 0.921 hours.
Optimizer stripped from D:\RIZ\Training\runs\lung_s1\weights\last.pt, 66.2MB
Optimizer stripped from D:\RIZ\Training\runs\lung_s1\weights\best.pt, 66.2MB

Validating D:\RIZ\Training\runs\lung_s1\weights\best.pt...
Ultralytics 8.4.53  Python-3.12.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
rt-detr-l summary: 310 layers, 31,998,125 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 8.1it/s 6.3s0.1s
                   all        404        166      0.576     0.0823     0.0086    0.00199
         

In [7]:
from pathlib import Path
from ultralytics import RTDETR

# ── CHECK YAML ────────────────────────────────────────────────
assert (
    'YAML_PATH' in globals() and
    Path(YAML_PATH).exists()
), 'Jalankan Cell 4 dulu!'

RUNS_DIR = OUTPUT_ROOT / 'runs'

RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ── HAPUS CACHE ULTRALYTICS ───────────────────────────────────
for cache_file in OUTPUT_ROOT.rglob("*.cache"):

    try:
        cache_file.unlink()
        print(f'Hapus cache: {cache_file}')

    except:
        pass

# ── BASE CONFIG ───────────────────────────────────────────────
BASE_CFG = dict(

    data=YAML_PATH,

    imgsz=640,

    batch=4,

    workers=0,

    cache=False,

    rect=False,

    optimizer='AdamW',

    weight_decay=1e-4,

    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.4,

    degrees=10.0,

    translate=0.1,

    scale=0.3,

    fliplr=0.5,

    flipud=0.0,

    copy_paste=0.0,

    mixup=0.0,

    multi_scale=False,

    dropout=0.2,

    plots=True,

    save=True,

    save_period=20,

    project=str(RUNS_DIR),

    exist_ok=True,

    seed=42,

    deterministic=True,

    verbose=True,
)

# ── LOAD CHECKPOINT TAHAP 1 ───────────────────────────────────
best_s1 = (
    RUNS_DIR /
    'lung_s1' /
    'weights' /
    'best.pt'
)

assert best_s1.exists(), \
    f'Checkpoint tidak ditemukan:\n{best_s1}'

print(f'Load checkpoint:\n{best_s1}')

# ── TRAIN TAHAP 2 ─────────────────────────────────────────────
print('\n[Tahap 2] Fine-tune semua layer — 270 epoch...')

model = RTDETR(str(best_s1))

model.train(

    **BASE_CFG,

    epochs=270,

    freeze=0,

    lr0=5e-5,

    lrf=0.01,

    warmup_epochs=10,

    mosaic=0.0,

    close_mosaic=0,

    patience=80,

    amp=True,

    name='lung_s2',
)

BEST_MODEL = str(
    RUNS_DIR /
    'lung_s2' /
    'weights' /
    'best.pt'
)

print('\n✓ Selesai')

print(f'Best model:\n{BEST_MODEL}')

Hapus cache: D:\RIZ\Training\split\labels\train.cache
Hapus cache: D:\RIZ\Training\split\labels\val.cache
Load checkpoint:
D:\RIZ\Training\runs\lung_s1\weights\best.pt

[Tahap 2] Fine-tune semua layer — 270 epoch...
New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.12.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\RIZ\Training\lung_usg.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=270, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.01, hsv_s=0.3,

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/270      4.06G      1.261     0.7663      1.785          3        640: 100% ━━━━━━━━━━━━ 391/391 2.8it/s 2:18<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.8it/s 6.6s0.1s
                   all        404        166   1.06e-05      0.039   1.54e-05   2.21e-06

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/270      4.26G      1.628     0.5005      1.975          4        640: 0% ──────────── 0/391  0.4s

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/270      4.26G     0.9471       1.17      1.074          3        640: 100% ━━━━━━━━━━━━ 391/391 3.1it/s 2:05<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.7it/s 6.6s0.1s
                   all        404        166   0.000232      0.212    0.00239    0.00054

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/270      4.26G      1.001      1.037       1.06          4        640: 0% ──────────── 0/391  0.3s

...

    250/270      4.27G     0.1222     0.2393     0.1089          3        640: 100% ━━━━━━━━━━━━ 391/391 3.3it/s 1:59<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.7it/s 6.6s0.1s
                   all        404        166      0.809      0.945      0.862      0.662

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    251/270      4.27G    0.07749     0.1607    0.05245          4        640: 0% ──────────── 0/391  0.3s

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    251/270      4.27G     0.1293     0.2408     0.1181          3        640: 100% ━━━━━━━━━━━━ 391/391 3.3it/s 2:00<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.7it/s 6.6s0.1s
                   all        404        166      0.844      0.942       0.87      0.666

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    252/270      4.27G          0   0.003816          0          4        640: 0% ──────────── 0/391  0.2s

d:\RIZ\Code\venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    252/270      4.27G     0.1272      0.235     0.1199          3        640: 100% ━━━━━━━━━━━━ 391/391 3.2it/s 2:01<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 7.7it/s 6.6s0.1s
                   all        404        166      0.837      0.938      0.866      0.664
EarlyStopping: Training stopped early as no improvement observed in last 80 epochs. Best results observed at epoch 172, best model saved as best.pt.
To update EarlyStopping(patience=80) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

252 epochs completed in 8.982 hours.
Optimizer stripped from D:\RIZ\Training\runs\lung_s2\weights\last.pt, 66.3MB
Optimizer stripped from D:\RIZ\Training\runs\lung_s2\weights\best.pt, 66.3MB

Validating D:\RIZ\Training\runs\lung_s2\weights\best.pt...
Ultralytics 8.4.53  Python-3.12.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
rt-detr-l summary: 310 layers,

## 5b. Resume Training (jika Colab disconnect)

In [ ]:
RUNS_DIR = str(OUTPUT_ROOT / 'runs')
for ckpt in sorted(Path(RUNS_DIR).glob('**/weights/*.pt')):
    print(f'  {ckpt}  ({ckpt.stat().st_size/1e6:.1f} MB)')

# Uncomment untuk resume:
# Stage 1
# last = str(Path(RUNS_DIR)/'lung_s1'/'weights'/'last.pt')

# Stage 2
last = str(Path(RUNS_DIR)/'lung_s2'/'weights'/'last.pt')
RTDETR(last).train(resume=True)

## 6. Evaluasi Model

In [8]:
# ── Evaluasi Model ────────────────────────────────────────────

from pathlib import Path
from ultralytics import RTDETR

BEST_MODEL = str(
    OUTPUT_ROOT /
    'runs' /
    'lung_s2' /
    'weights' /
    'best.pt'
)

YAML_PATH = str(
    OUTPUT_ROOT /
    'lung_usg.yaml'
)

# ── CHECK FILE ────────────────────────────────────────────────
assert Path(BEST_MODEL).exists(), \
    f'BEST_MODEL tidak ditemukan:\n{BEST_MODEL}'

assert Path(YAML_PATH).exists(), \
    f'YAML_PATH tidak ditemukan:\n{YAML_PATH}'

print(f'BEST_MODEL:\n{BEST_MODEL}')
print(f'YAML_PATH :\n{YAML_PATH}')

# ── LOAD MODEL ────────────────────────────────────────────────
model_eval = RTDETR(BEST_MODEL)

# ── Evaluasi VAL & TEST ───────────────────────────────────────
for split in ['val', 'test']:

    print('\n' + '='*55)

    print(f'Evaluasi {split.upper()}')

    print('='*55)

    m = model_eval.val(

        data=YAML_PATH,

        split=split,

        imgsz=640,

        batch=4,

        workers=0,

        conf=0.25,

        iou=0.5,

        plots=True,

        verbose=True,
    )

    print(f'\nmAP50     : {m.box.map50:.4f}')

    print(f'mAP50-95  : {m.box.map:.4f}')

    print(f'Precision : {m.box.mp:.4f}')

    print(f'Recall    : {m.box.mr:.4f}')

    # ── Per-class AP ──────────────────────────────────────────
    if hasattr(m.box, 'ap_class_index'):

        print('\nPer-class AP50:')

        for i, cid in enumerate(m.box.ap_class_index):

            ap = (
                m.box.ap50[i]
                if hasattr(m.box, 'ap50')
                else 0
            )

            print(
                f'  {CLASS_NAMES[cid]:<30} '
                f'AP50={ap:.4f}'
            )

# ── Variance Test ─────────────────────────────────────────────
print('\n' + '='*55)

print('Variance test (3 seed)')

print('='*55)

maps = []

for seed in [42, 123, 777]:

    torch.manual_seed(seed)

    np.random.seed(seed)

    mv = RTDETR(BEST_MODEL).val(

        data=YAML_PATH,

        split='test',

        imgsz=640,

        batch=4,

        workers=0,

        verbose=False,
    )

    maps.append(mv.box.map50)

    print(
        f'  seed={seed:<3} '
        f'mAP50={mv.box.map50:.4f}'
    )

print(
    f'\nFinal mAP50: '
    f'{np.mean(maps):.4f} '
    f'± {np.std(maps):.4f}'
)

BEST_MODEL:
D:\RIZ\Training\runs\lung_s2\weights\best.pt
YAML_PATH :
D:\RIZ\Training\lung_usg.yaml

Evaluasi VAL
Ultralytics 8.4.53  Python-3.12.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
rt-detr-l summary: 310 layers, 31,998,125 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 540.672.6 MB/s, size: 41.4 KB)
val: Scanning D:\RIZ\Training\split\labels\val.cache... 404 images, 243 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 404/404  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 9.5it/s 10.6s0.1s
                   all        404        166      0.841      0.957      0.902      0.699
               kavitas         12         12      0.693      0.917      0.848      0.673
                pola_b         33         33       0.84          1      0.945        0.7
           konsolidasi         68         72      0.883      0.972      0.931      0.718
      penebala

## 7. GUI Gradio

In [9]:
import gradio as gr
import matplotlib.patches as patches

# ── Color & Info ──────────────────────────────────────────────
CLASS_COLORS = [
    '#DC3232',
    '#3278DC',
    '#32A852',
    '#DCA032',
    '#9632DC',
    '#DC6432',
    '#00BFA5'
]

CLASS_INFO = {

    'kavitas':
        'Rongga berisi udara/cairan. '
        'Sering terkait TB atau abses.',

    'pola_b':
        'B-lines vertikal hiperreflektif — '
        'penebalan septum interlobular.',

    'konsolidasi':
        'Area paru terisi cairan/jaringan padat. '
        'Khas pada pneumonia.',

    'penebalan_pleura':
        'Pleura menebal akibat peradangan '
        'atau efusi kronis.',

    'bullae':
        'Kantung udara berdinding tipis. '
        'Terkait emfisema.',

    'infiltrat_terkonsolidasi':
        'Gabungan infiltrat dan konsolidasi. '
        'Infeksi luas atau ARDS.',

    'pleura_ireguler':
        'Garis pleura tidak rata/terputus. '
        'Tanda awal pneumothorax atau kontusio.'
}

# ── Cache model ───────────────────────────────────────────────
_cache = {}

def get_model(p):

    if p not in _cache:
        _cache[p] = RTDETR(p)

    return _cache[p]

# ── Preprocessing ─────────────────────────────────────────────
def prep(img):

    g = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    e = cv2.createCLAHE(
        2.0,
        (8,8)
    ).apply(g)

    f = cv2.bilateralFilter(
        e,
        9,
        75,
        75
    )

    f = cv2.resize(
        f,
        (640,640),
        interpolation=cv2.INTER_AREA
    )

    return cv2.cvtColor(
        f,
        cv2.COLOR_GRAY2BGR
    )

# ── Detection ─────────────────────────────────────────────────
def detect(
    image,
    model_path,
    conf,
    do_prep,
    sample_name=None
):

    if image is None:

        return (
            None,
            'Upload gambar dulu',
            '',
            ''
        )

    if not Path(model_path).exists():

        return (
            None,
            f'Model tidak ada:\n{model_path}',
            '',
            ''
        )

    # filename
    fname = (
        sample_name
        if sample_name
        else 'uploaded_image'
    )

    # PIL → BGR
    bgr = cv2.cvtColor(
        np.array(image),
        cv2.COLOR_RGB2BGR
    )

    inp = prep(bgr) if do_prep else bgr

    if not do_prep:

        inp = cv2.resize(
            inp,
            (640,640),
            interpolation=cv2.INTER_AREA
        )

    # temp
    tmp = OUTPUT_ROOT / '_temp_usg.jpg'

    cv2.imwrite(str(tmp), inp)

    # inference
    res = get_model(model_path).predict(

        str(tmp),

        conf=conf,

        imgsz=640,

        verbose=False,
    )

    # plot
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13,5)
    )

    fig.patch.set_facecolor('#12122a')

    for ax, im, ttl in [

        (
            axes[0],
            cv2.cvtColor(
                bgr,
                cv2.COLOR_BGR2RGB
            ),
            'Original'
        ),

        (
            axes[1],
            cv2.cvtColor(
                inp,
                cv2.COLOR_BGR2RGB
            ),
            'Preprocessed'
            if do_prep
            else 'Input'
        )
    ]:

        ax.imshow(im)

        ax.set_title(
            ttl,
            color='w',
            fontsize=10
        )

        ax.axis('off')

        ax.set_facecolor('#12122a')

    dets = []

    seen = []

    # bbox
    for r in res:

        for box in r.boxes:

            x1,y1,x2,y2 = (
                box.xyxy[0]
                .cpu()
                .numpy()
            )

            cvf = float(box.conf[0])

            cid = int(box.cls[0])

            nm = (
                CLASS_NAMES[cid]
                if cid < NC
                else f'cls{cid}'
            )

            cl = (
                CLASS_COLORS[cid]
                if cid < len(CLASS_COLORS)
                else '#FFFFFF'
            )

            axes[1].add_patch(

                patches.FancyBboxPatch(

                    (x1,y1),

                    x2-x1,

                    y2-y1,

                    boxstyle='round,pad=2',

                    lw=2,

                    edgecolor=cl,

                    facecolor='none'
                )
            )

            axes[1].text(

                x1,

                max(y1-8,0),

                f'{nm} {cvf:.0%}',

                color='white',

                fontsize=8,

                fontweight='bold',

                bbox=dict(
                    facecolor=cl,
                    alpha=0.8,
                    pad=2,
                    edgecolor='none'
                )
            )

            dets.append(
                (
                    nm,
                    cvf,
                    (x1,y1,x2,y2)
                )
            )

            if nm not in seen:
                seen.append(nm)

    plt.tight_layout(pad=1)

    # no detection
    if not dets:

        plt.close('all')

        return (
            fig,
            'Tidak ada kelainan terdeteksi.\n'
            'Coba turunkan threshold.',
            '',
            fname
        )

    # result text
    txt = '\n'.join([

        f'{i+1}. '
        f'{n.upper()}  '
        f'{c:.1%}  '
        f'bbox({x1:.0f},{y1:.0f})'
        f'→({x2:.0f},{y2:.0f})'

        for i,(n,c,(x1,y1,x2,y2))
        in enumerate(dets)
    ])

    # clinical info
    info = (
        'INFO KLINIS:\n\n'
        +
        '\n'.join([

            f'• {n.upper()}\n'
            f'  {CLASS_INFO.get(n,"")}\n'

            for n in seen
        ])
    )

    plt.close('all')

    return (
        fig,
        txt,
        info,
        fname
    )

# ── Confidence chart ──────────────────────────────────────────
def chart(
    image,
    model_path,
    conf,
    do_prep,
    sample_name=None
):

    if (
        image is None or
        not Path(model_path).exists()
    ):
        return None

    bgr = cv2.cvtColor(
        np.array(image),
        cv2.COLOR_RGB2BGR
    )

    inp = prep(bgr) if do_prep else bgr

    if not do_prep:

        inp = cv2.resize(
            inp,
            (640,640),
            interpolation=cv2.INTER_AREA
        )

    tmp = OUTPUT_ROOT / '_temp_usg2.jpg'

    cv2.imwrite(str(tmp), inp)

    res = get_model(model_path).predict(

        str(tmp),

        conf=0.01,

        imgsz=640,

        verbose=False
    )

    best = {
        i:0.0
        for i in range(NC)
    }

    for r in res:

        for box in r.boxes:

            c = int(box.cls[0])

            v = float(box.conf[0])

            if v > best.get(c,0):
                best[c] = v

    fig, ax = plt.subplots(
        figsize=(7,3.5)
    )

    fig.patch.set_facecolor('#12122a')

    ax.set_facecolor('#12122a')

    ax.barh(

        [
            n.replace('_','\n')
            for n in CLASS_NAMES
        ],

        [
            best[i]
            for i in range(NC)
        ],

        color=CLASS_COLORS,

        alpha=0.85,

        height=0.6
    )

    ax.axvline(

        conf,

        color='white',

        ls='--',

        lw=1,

        alpha=0.6,

        label=f'Threshold {conf:.0%}'
    )

    ax.set_xlim(0,1.05)

    ax.set_xlabel(
        'Confidence',
        color='white',
        fontsize=9
    )

    ax.tick_params(
        colors='white',
        labelsize=8
    )

    ax.spines[
        ['top','right','bottom']
    ].set_visible(False)

    ax.spines['left'].set_color('#444')

    ax.legend(
        facecolor='#12122a',
        labelcolor='white',
        fontsize=8
    )

    plt.tight_layout()

    plt.close('all')

    return fig

# ── Default model ─────────────────────────────────────────────
MDL = (
    BEST_MODEL
    if 'BEST_MODEL' in globals()
    else str(
        OUTPUT_ROOT /
        'runs' /
        'lung_s2' /
        'weights' /
        'best.pt'
    )
)

# ── UI ────────────────────────────────────────────────────────
with gr.Blocks(

    theme=gr.themes.Soft(),

    title='USG Paru'

) as demo:

    gr.HTML(
        '<h2 style="text-align:center;color:#4a9eff">'
        'Deteksi Penyakit Paru — USG'
        '</h2>'
    )

    gr.HTML(
        '<p style="text-align:center;color:#888">'
        'RT-DETR · 7 Kelas'
        '</p>'
    )

    with gr.Row():

        # left
        with gr.Column(scale=1):

            img_in = gr.Image(
                label='Upload Citra USG',
                type='pil',
                height=280
            )

            mdl_box = gr.Textbox(
                label='Path Model (.pt)',
                value=MDL
            )

            csl = gr.Slider(
                0.1,
                0.9,
                value=0.25,
                step=0.05,
                label='Confidence Threshold'
            )

            ppbox = gr.Checkbox(
                label='CLAHE + bilateral filter',
                value=True
            )

            file_box = gr.Textbox(
                label='Nama File',
                interactive=False
            )

            btn = gr.Button(
                'Deteksi',
                variant='primary',
                size='lg'
            )

            lgnd = ''.join([

                f'<span style="background:{c};'
                f'color:#fff;'
                f'padding:2px 8px;'
                f'border-radius:4px;'
                f'font-size:11px;'
                f'margin:2px;'
                f'display:inline-block">'
                f'{n}'
                f'</span>'

                for n,c
                in zip(
                    CLASS_NAMES,
                    CLASS_COLORS
                )
            ])

            gr.HTML(
                f'<div style="margin-top:8px">'
                f'{lgnd}'
                f'</div>'
            )

        # right
        with gr.Column(scale=2):

            with gr.Tabs():

                with gr.Tab('Deteksi'):

                    img_out = gr.Plot()

                with gr.Tab('Confidence'):

                    ch_out = gr.Plot()

            with gr.Row():

                res_box = gr.Textbox(
                    label='Hasil Deteksi',
                    lines=6,
                    interactive=False
                )

                info_box = gr.Textbox(
                    label='Info Klinis',
                    lines=6,
                    interactive=False
                )

    # examples
    test_imgs = list(
        (
            OUTPUT_ROOT /
            'split' /
            'images' /
            'test'
        ).glob('*.jpg')
    )[:4]

    if test_imgs:

        ex_data = []

        for p in test_imgs:

            ex_data.append([
                str(p),
                p.name
            ])

        gr.Examples(

            examples=ex_data,

            inputs=[
                img_in,
                file_box
            ],

            label='Contoh test set'
        )

    ins = [
        img_in,
        mdl_box,
        csl,
        ppbox,
        file_box
    ]

    btn.click(

        detect,

        ins,

        [
            img_out,
            res_box,
            info_box,
            file_box
        ]

    ).then(

        chart,

        ins,

        [ch_out]
    )

    csl.release(

        detect,

        ins,

        [
            img_out,
            res_box,
            info_box,
            file_box
        ]
    )

# ── Launch ────────────────────────────────────────────────────
demo.launch(

    share=True,

    server_name='0.0.0.0',

    allowed_paths=[
        str(
            OUTPUT_ROOT /
            'split' /
            'images' /
            'test'
        )
    ]
)

d:\RIZ\Code\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_2648\2580093379.py:480: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://3de9b438d82fb88ba0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
